In [2]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import tensorflow as tf
#from tensorflow.keras import layers, models, losses
#from tensorflow.keras.callbacks import ModelCheckpoint
from keras import layers, models, losses, regularizers
from keras.models import load_model
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print("GPU Trovate:", len(tf.config.list_physical_devices('GPU')))
for gpu in tf.config.list_physical_devices('GPU'):
    print("Nome:", gpu.name)

# 1. SETUP MEMORIA: DEVE ESSERE LA PRIMA COSA IN ASSOLUTO
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memoria GPU configurata in modalità dinamica.")
    except RuntimeError as e:
        print("Errore GPU:", e)

# 2. ESORCISMO DELLA RAM (Uccide i vecchi modelli in memoria)
tf.keras.backend.clear_session()

GPU Trovate: 1
Nome: /physical_device:GPU:0
Memoria GPU configurata in modalità dinamica.


In [3]:
def embedded_summary(model, input_shape=(1, 120, 18), is_int8=False):
    total_params = model.count_params()
    
    # 1. Calcolo FLASH (4 byte per Float32, 1 byte per INT8)
    bytes_per_param = 1 if is_int8 else 4
    estimated_flash_kb = (total_params * bytes_per_param) / 1024
    
    # 2. Calcolo SRAM (Tensor Arena) con logica Adiacente (Buffer Reuse)
    bytes_per_activation = 1 if is_int8 else 4
    max_adjacent_ram_kb = 0
    
    # Memoria occupata dal layer precedente (inizializzata con la dimensione dell'input)
    previous_layer_size = (np.prod(input_shape) * bytes_per_activation) / 1024
    
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            # Per i layer come "Concatenate" che potrebbero avere output multipli/strani
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        current_layer_size = (num_elements * bytes_per_activation) / 1024
        
        # IL FIX È QUI: Sommiamo il layer precedente e il layer corrente!
        # È il momento esatto in cui TFLM consuma più RAM durante l'esecuzione di questo layer.
        current_peak = previous_layer_size + current_layer_size
        
        if current_peak > max_adjacent_ram_kb:
            max_adjacent_ram_kb = current_peak
            
        previous_layer_size = current_layer_size

    print("============================================")
    mode_str = "INT8 (Quantizzato)" if is_int8 else "FLOAT32 (Training)"
    print(f"   REPORT REQUISITI ESP32-S3 [{mode_str}]   ")
    print("============================================")
    print(f" Memoria FLASH stimata : ~{estimated_flash_kb:.2f} KB  (Limite: 800 KB)")
    print(f" Memoria SRAM stimata  : ~{max_adjacent_ram_kb:.2f} KB (Limite: 300 KB)")
    print("============================================\n")

In [10]:
import itertools

PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost =  coords_cost_norm +  4.0 * mask_cost_norm 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost =  coords_cost_norm +  4.0 * mask_cost_norm
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

In [5]:
# ==============================================================================
# 1. DATA ENGINE V12 (Caricamento Globale in RAM, LOG1P per segnali deboli) 
# ==============================================================================

def load_and_process_all_files(file_list, alpha=0.002):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   # Shape: (T, 6, 3, 120, 2)
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        
        # TRUCCO MAGISTRALE: Shape (T, 6, 120, 3)
        mag_reshaped = np.transpose(mag, (0, 1, 3, 2)) 
        
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        # EMA
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
            
        # ====================================================================
        # FIX FISICA RADAR: Compressione logaritmica (salva le persone a 6m)
        # ====================================================================
        decluttered = np.log1p(decluttered)
        
        # Flatten delle coordinate
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato.")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)

    return X, Y

# ==============================================================================
# 2. SPLIT E NORMALIZZAZIONE RIGOROSA
# ==============================================================================
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]

tutti_i_file = glob.glob("dataset/data/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

print("\n--- PREPARAZIONE TRAINING SET ---")
X_train_raw, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val_raw, Y_val = load_and_process_all_files(val_files)

print("\n--- CALCOLO STATISTICHE E NORMALIZZAZIONE ---")
train_mean = np.mean(X_train_raw)
train_std = np.std(X_train_raw)

print(f"!!! VALORI DA SALVARE E HARDCODARE NEL TUO CODE.PY !!!")
print(f"ATTENZIONE: Questi numeri sono cambiati per via del log1p!")
print(f"MEAN: {train_mean:.6f}")
print(f"STD:  {train_std:.6f}")

X_train = (X_train_raw - train_mean) / (train_std + 1e-7)
X_val = (X_val_raw - train_mean) / (train_std + 1e-7)

del X_train_raw
del X_val_raw


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 18 file...
File 1/18 processato.
File 2/18 processato.
File 3/18 processato.
File 4/18 processato.
File 5/18 processato.
File 6/18 processato.
File 7/18 processato.
File 8/18 processato.
File 9/18 processato.
File 10/18 processato.
File 11/18 processato.
File 12/18 processato.
File 13/18 processato.
File 14/18 processato.
File 15/18 processato.
File 16/18 processato.
File 17/18 processato.
File 18/18 processato.

--- PREPARAZIONE VALIDATION SET ---
Inizio caricamento ed EMA Decluttering di 6 file...
File 1/6 processato.
File 2/6 processato.
File 3/6 processato.
File 4/6 processato.
File 5/6 processato.
File 6/6 processato.

--- CALCOLO STATISTICHE E NORMALIZZAZIONE ---
!!! VALORI DA SALVARE E HARDCODARE NEL TUO CODE.PY !!!
ATTENZIONE: Questi numeri sono cambiati per via del log1p!
MEAN: 1.915614
STD:  1.087612


In [7]:
# ==============================================================================
# 3. ARCHITETTURA EEAI-NET V12 "CROSS-TALK" (Fix Geometrico e Bounding)
# ==============================================================================

def build_eeai_model_v12_crosstalk(n_radars=6, n_antennas=3, n_bins=120):
    inputs = layers.Input(shape=(n_radars, n_bins, n_antennas), name="radar_input")

    # 1. ESTRAZIONE INDIPENDENTE E FUSIONE ANTENNE
    # Usiamo Conv2D standard (NON separable) per fondere la fase delle 3 antenne!
    x = layers.Conv2D(32, (1, 5), padding='same')(inputs)
    x = layers.LeakyReLU(alpha=0.1)(x) # Salva i segnali negativi dopo la standardizzazione
    x = layers.MaxPooling2D((1, 2))(x)  # Shape: (6, 60, 32)
    
    # 2. CROSS-RADAR FUSION (Fonde i 6 radar in un'unica mappa spaziale)
    # Kernel (6, 5) con padding valid fa collassare i 6 radar!
    x = layers.Conv2D(64, (6, 5), padding='valid')(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.MaxPooling2D((1, 2))(x)  # Shape: (1, 28, 64)
    
    # 3. SQUEEZE LAYER
    x = layers.Conv2D(16, (1, 1), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.1)(x)  # Shape: (1, 28, 16)
    
    # 4. FUSIONE GLOBALE
    x = layers.Flatten(name="flatten_spatial_map")(x)  
    x = layers.Dropout(0.3, name="heavy_spatial_dropout")(x) 
    
    x = layers.Dense(128)(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Dropout(0.2)(x) 
    
    common_feat = layers.Dense(64)(x)
    common_feat = layers.LeakyReLU(alpha=0.1)(common_feat)

    # 5. COORDINATE BOUNDING (Forza la regressione dentro i limiti della stanza)
    raw_coords = layers.Dense(8, activation='sigmoid', name="raw_coords_head")(common_feat)
    
    # IL FIX E' QUI: Aggiunte parentesi quadre esterne. Shape diventa (1, 8) 
    # così si adatta automaticamente a (Batch, 8) in moltiplicazione!
    room_scale = tf.constant([[4.8, 7.2, 4.8, 7.2, 4.8, 7.2, 4.8, 7.2]], dtype=tf.float32)
    
    coords_output = layers.Multiply(name="coords_head")([raw_coords, room_scale])
    
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V12_CrossTalk")

model_v12_final = build_eeai_model_v12_crosstalk()

if 'embedded_summary' in globals():
    embedded_summary(model_v12_final) # Sarà meno di 600 KB Float32!

   REPORT REQUISITI ESP32-S3 [FLOAT32 (Training)]   
 Memoria FLASH stimata : ~506.11 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~0.00 KB (Limite: 300 KB)



In [ ]:
# ==============================================================================
# 4. GENERATORE CON RADAR DROPOUT E FINE TUNING
# ==============================================================================

class RadarAugmentGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, Y, batch_size=32, drop_prob=0.35, max_drop=2):
        self.X = X
        self.Y = Y
        self.batch_size = batch_size
        self.drop_prob = drop_prob   
        self.max_drop = max_drop     
        self.indices = np.arange(len(self.X))
        np.random.shuffle(self.indices)
        
    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))
        
    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx+1) * self.batch_size]
        X_batch = self.X[batch_idx].copy() 
        Y_batch = self.Y[batch_idx]
        
        for i in range(len(X_batch)):
            if np.random.rand() < self.drop_prob:
                num_drop = np.random.randint(1, self.max_drop + 1)
                drop_idx = np.random.choice(6, num_drop, replace=False)
                X_batch[i, drop_idx, :, :] = 0.0 
                
        return X_batch, Y_batch
        
    def on_epoch_end(self):
        np.random.shuffle(self.indices)

train_generator = RadarAugmentGenerator(X_train, Y_train, batch_size=32)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001) #da 0.0005 a 0.001 per vedere se accelera il training senza peggiorare la convergenza


model_v12_final.compile(
    optimizer=optimizer,
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_v12 = ModelCheckpoint("eeai_best_model_v12.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)

print("\n--- INIZIO ADDESTRAMENTO V12 CROSSTALK CON RADAR DROPOUT E LOG1P ---")
history_v12_final = model_v12_final.fit(
    train_generator,                 
    validation_data=(X_val, Y_val),  
    epochs=300,
    callbacks=[checkpoint_v12, reduce_lr, early_stop], 
    verbose=1
)


--- INIZIO ADDESTRAMENTO V12 CROSSTALK CON RADAR DROPOUT E LOG1P ---
Epoch 1/300


/home/marco/yes/envs/edge_ai_env/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1783074093.628251   19074 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1421252__.47


3153/4219 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - hungarian_mask_acc: 0.9891 - hungarian_rmse_metres: 0.4752 - loss: 0.4701

I0000 00:00:1783074104.116296   19076 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1421252__.47


4206/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - hungarian_mask_acc: 0.9890 - hungarian_rmse_metres: 0.4768 - loss: 0.4746

W0000 00:00:1783074108.664900   19007 cpu_allocator_impl.cc:82] Allocation of 388800000 exceeds 10% of free system memory.
W0000 00:00:1783074109.049601   19007 cpu_allocator_impl.cc:82] Allocation of 388800000 exceeds 10% of free system memory.
I0000 00:00:1783074109.499603   19072 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1442897__.20
I0000 00:00:1783074111.573750   19072 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1442897__.20



Epoch 1: val_loss improved from None to 1.01921, saving model to eeai_best_model_v12.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - hungarian_mask_acc: 0.9884 - hungarian_rmse_metres: 0.4817 - loss: 0.4904 - val_hungarian_mask_acc: 0.9607 - val_hungarian_rmse_metres: 0.4916 - val_loss: 1.0192 - learning_rate: 0.0010
Epoch 2/300
4218/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - hungarian_mask_acc: 0.9884 - hungarian_rmse_metres: 0.4884 - loss: 0.4998
Epoch 2: val_loss improved from 1.01921 to 0.96157, saving model to eeai_best_model_v12.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - hungarian_mask_acc: 0.9882 - hungarian_rmse_metres: 0.4862 - loss: 0.4993 - val_hungarian_mask_acc: 0.9598 - val_hungarian_rmse_metres: 0.4808 - val_loss: 0.9616 - learning_rate: 0.0010
Epoch 3/300
4207/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - hungarian_mask_acc: 0.9884 - hungarian_rmse_metres: 0.4827 - loss: 0.4949
Epoch 3: val_loss improved from 0.96157 to 0.82825, saving model to eeai_best_model_v12.

In [ ]:
# ==============================================================================
# VISUALIZZATORE 4.0 (Compatibile con V10/V11 "Condivise" Shape: 6, 120, 3)
# ==============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from tensorflow.keras.models import load_model

file_target = "dataset/data/window_000011.npz"
# Seleziona qui il nome del modello che vuoi testare
NOME_MODELLO = "eeai_best_model_v12.keras" 

if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print(f"Elaborazione filtri e previsioni in corso ({NOME_MODELLO})...")
    
    # 1. Calcolo Magnitudo: (T, 6, 3, 120)
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
    
    # ==========================================
    # FIX GEOMETRICO: Trasposizione per V10/V11
    # ==========================================
    # Da (T, 6, 3, 120) a (T, 6, 120, 3)
    mag_reshaped = np.transpose(mag, (0, 1, 3, 2))
    
    decluttered = np.zeros_like(mag_reshaped)
    bg = np.copy(mag_reshaped[0])
    alpha = 0.002
    
    for t in range(T):
        bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag_reshaped[t] - bg)
    
    print("Applicazione Standardizzazione Globale...")
    # ATTENZIONE: Assicurati che questi siano i valori del training su cui ha imparato il modello!
    MEAN_TRAINING = 1.915614
    STD_TRAINING = 1.087612

    decluttered = (decluttered - MEAN_TRAINING) / (STD_TRAINING + 1e-7)

    print(f"Caricamento dei pesi migliori dal file {NOME_MODELLO} ...")
    
    # Caricamento del modello 
    model_final = load_model(
        NOME_MODELLO,
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_final.predict(decluttered, verbose=0)
    
    # Slicing invariato e corretto: prime 8 coord, ultime 4 maschere
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            # Titolo aggiornato dinamicamente
            modello_str = NOME_MODELLO.split('.')[0]
            ax.set_title(f"Radar {modello_str.upper()} | Frame: {frame_idx}/{T-1} | Window: 11", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri e previsioni in corso (eeai_best_model_v12.keras)...
Applicazione Standardizzazione Globale...
Caricamento dei pesi migliori dal file eeai_best_model_v12.keras ...


I0000 00:00:1783075202.054066   19076 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3397542__.5
I0000 00:00:1783075202.501508   19071 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3398121__.5


Dati pronti! Inizializzazione Radar...
